## Initialization

### Imports

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    # TypeVar,
    # Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log
import shutil
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from scipy.interpolate import make_interp_spline
from pint import Quantity

from data_processing import processing as proc
from data_processing import loading as load
from data_processing import types as proc_types
from data_processing import helpers
# from data_processing.paths import (
#     get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    # NonReactorDataframeColumn,
    # SliceFitDataframeColumn,
    EnergyColumn,
    get_df_col
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
# from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing.processing.neutron_window_strategy.strategy_factory \
    import NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy \
    import AbstractNeutronStrategy
# from data_processing.helpers import (
#     # get_input_with_default,
#     # get_input_required,
#     # input_experiment_ids,
#     stop,
#     get_midpoints_from_bins
# )


### Functions

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

In [ ]:
def bin_non_neutron_data(df, time_bins, data_col, selected_cols):
    start_time = time_bins[0]
    df = get_time_cut(df, 'Time', time_bins)

    binned_df = df.groupby("Time Bin", as_index=False)[data_col] \
        .agg(['mean', 'std']) \
        .copy()
    binned_df.columns = selected_cols
    binned_df['Bin midpoint'] = binned_df.index.to_series() \
        .apply(lambda x: x.mid)
    binned_df = bin_midpoint_time_to_seconds(binned_df, start_time)

    return binned_df

In [ ]:
def bin_midpoint_time_to_seconds(df, start_time):
    bin_mid_col = df[BinningDataframeColumn.BIN_MIDPOINT.value]
    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
    zeroed_midpoint = pd.to_datetime(bin_mid_col) - start_time
    df[bin_time_col_name] = zeroed_midpoint.dt.total_seconds()
    return df

In [ ]:
def get_time_cut(df, time_tag_col, time_bins):
    timetag_cut = pd.cut(df[time_tag_col], bins=time_bins)
    df[BinningDataframeColumn.TIME_BIN.value] = timetag_cut
    return df

In [ ]:
# fns ask questions, then generate strategy using factory

CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]


def get_nasa_loading_settings(
    calib_key: CalibrationKey
) -> str:
    left_border_type = helpers.get_input_with_default(
        """\
Which left border calculation do you want to use?
1: original left border (0.1966 MeVee)
2: newer left border (~0.1866 MeVee)
3: CAEN lower limit (0.050 MeVee) (default)
Press Enter for default
""",
        3,
        int
    )
    border_key: NasaBorderKey = (
        ExperimentDataKey.NASA_BORDERS if left_border_type == 1 
        else ExperimentDataKey.NASA_BORDERS_RECALC
    )
    file_name_prefix = f"{calib_key.value}_{border_key.value}"
    return file_name_prefix


def get_n_distro_loading_settings(
    calib_key: CalibrationKey
) -> str:
    file_name_prefix = f"{calib_key.value}_{ExperimentDataKey.N_WINDOW_BORDERS.value}"
    return file_name_prefix


def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> proc_types.NasaGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = helpers.get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = helpers.get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = helpers.get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee)
2: newer (~0.1866 MeVee)
3: detector lower limit (0.050 MeVee) (default)
or press Enter for default
""",
            3,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = (
                ExperimentDataKey.NASA_BORDERS 
                if existing_left_border_version_input == 1 
                else ExperimentDataKey.NASA_BORDERS_RECALC
            )
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = load.get_neutron_window_paths(
                file_name_prefix=file_name_prefix)
            left_border, _ = load.load_side_borders(
                side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        elif existing_left_border_version_input == 3:
            lower_energy_bound = 0.05
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = helpers.get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.050)
""",
            0.050,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def get_n_distro_generation_settings(
) -> proc_types.NeutronDistributionGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (3)
""",
        3,
        float
    )
    settings = proc_types.NeutronDistributionGenerationSettings(
        sigma=sigma
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: proc.NeutronStrategyFactory,
    window_type: proc_types.WindowType,
    loading: bool,
    settings: proc_types.NeutronWindowSettings
) -> Callable[[], AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data


In [ ]:
# def relative_rmse(x: pd.Series | float, x_err: pd.Series | float, y: pd.Series | float, y_err: pd.Series | float) -> pd.Series | float:
def relative_rmse(values: list[tuple[pd.Series | float, pd.Series | float]]) -> pd.Series | float:
    rel_sq_values = [relative_square_error(x, x_err) for x, x_err in values]
    # rel_sq_x = relative_square_error(x, x_err)
    # rel_sq_y = relative_square_error(y, y_err)
    # rel_sq_sum = rel_sq_x + rel_sq_y
    rel_sq_sum = sum(rel_sq_values)
    if isinstance(rel_sq_sum, pd.Series):
        return rel_sq_sum.pow(1./2)
    else:
        return rel_sq_sum ** (1./2)


def relative_square_error(x: pd.Series | float, x_err: pd.Series | float) -> pd.Series | float:
    # divide x_err by x
    # square it
    # return
    rel_err = x_err / x
    if isinstance(rel_err, pd.Series):
        return rel_err.pow(2).fillna(0)
    else:
        return rel_err ** 2

In [ ]:
def correct_raw_signals(
    raw_signals_df: pd.DataFrame,
    baseline_idx_range: int = 40,
    baseline_offset: float = 0,
    max_adc: int = 16367,
    use_max_adc: bool = False
) -> pd.DataFrame:
    offset = int(baseline_offset * max_adc)
    signals_np = raw_signals_df.to_numpy()
    
    if use_max_adc:
        baselines = max_adc
    else:
        baselines = signals_np[
            :, :baseline_idx_range
        ].mean(axis=1).reshape(-1, 1)
    
    signals_np = -signals_np + baselines + offset
    corrected_signals = pd.DataFrame(
        signals_np,
        index=raw_signals_df.index,
        columns=raw_signals_df.columns
    )
    return corrected_signals

In [ ]:
def get_psd_adc_histogram(
    df: pd.DataFrame,
    adc_col: str = DetectorDataframeColumn.ENERGY.value,
    adc_width: float = 420,
    adc_bins: np.ndarray | None = None,
    psd_bin_count: int = 100,
    psd_min: float = 0.0,
    psd_max: float = 0.5
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x = df[adc_col]
    y = get_df_col(df, DetectorDataframeColumn.PSD)

    within_psd = y.between(psd_min, psd_max)
    x = x[within_psd == True].copy()
    y = y[within_psd == True].copy()

    if adc_bins is not None:
        x_bins = adc_bins
    else:
        x_bins: np.ndarray = np.linspace(
            0, x.max(), int(x.max() / adc_width) + 1
        )
    print(f"Energy width = {x_bins[1]-x_bins[0]} ADC")
    y_bins: np.ndarray = np.linspace(psd_min, psd_max, psd_bin_count + 1)

    Z, xe, ye = np.histogram2d(x, y, bins=[x_bins, y_bins])
    return Z, xe, ye

## Data Loading

### Loading Params

In [ ]:
experiment_ids = ["TB-26"]

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
# calib_input = get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
# settings = get_nasa_generation_settings(calib_key)
window_offset = 0.2
sigma = 5
lower_energy_bound = 0.05
recalc_lower_bound = False
settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

In [ ]:
# experiment_ids = input_experiment_ids()

In [ ]:
# # more here?
# experiment_neutron_data: ExperimentNeutronData = {
#     exp_id: {}
#     for exp_id in experiment_ids
# }

In [ ]:
# calib_input = get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )

# is_new_calibration = calib_input.lower() == "y"
# calibrated_energy_column: EnergyColumn = (
#     DetectorDataframeColumn.RECALIBRATED_ENERGY
#     if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
# )
# calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
# strategy_factory = proc.NeutronStrategyFactory()
# settings = get_nasa_generation_settings(calib_key)
# factory_fn = make_strategy_factory_fn(
#     strategy_factory, "nasa", False, settings)
# experiment_neutron_data = make_strategy_for_experiments(
#     experiment_neutron_data, factory_fn)

### Loading and Initial Processing

In [ ]:
figure_data = {k: {} for k in ["a", "b", "c", "d"]}

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load.load_parquet_psd(exp_id, with_flags=True)
    exp_data["signals_df"] = load.load_parquet_signals(exp_id)

In [ ]:
# Ensure that index matches between signals and CAEN data
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    signals_df = exp_data["signals_df"]
    
    unclassified_index: pd.Index = unclassified_df.index
    signals_index: pd.Index = signals_df.index
    clean_index = unclassified_index.intersection(signals_index)
    
    unclassified_df = unclassified_df.loc[clean_index]
    signals_df = signals_df.loc[clean_index]
    
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df
    exp_data["signals_df"] = signals_df

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = load.calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = proc.recalibrate(unclassified_df, proc.Detector.ZERO)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

## Data Processing

### Pulse Processing

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    signals_df = exp_data["signals_df"].astype("int32")
    
    signals_df = correct_raw_signals(signals_df)
    heights = signals_df.max(axis=1)
    print(heights.max())
    signals_df.columns = signals_df.columns.map(int)
    
    psd_report["height"] = heights
    exp_data["signals_df"] = signals_df
    exp_data[ExperimentDataKey.UNCLASSIFIED] = psd_report

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
adc_width = 20

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = get_psd_adc_histogram(
        psd_report,
        # adc_col="height",
        adc_width=adc_width
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
# TODO get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

    # # Default
    # default_bounds: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000)
    # )

    # bounds_a: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000)
    # )

    # bounds_b: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000)
    # )

    # # Ranged Example
    # bounds = [
    #     ((0, 60), bounds_a),
    # ]

    df, df_err = proc.scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style="peak_finder",
        # default_bounds,
        # bounds=bounds,
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = proc.find_failed_slices(df, exp_id)

    if bad_slice_indexes is not None:
        exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
        exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        # exp_data['fom_results'] = df
        exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    helpers.stop()

In [ ]:
# get borders from strategy
for exp_id, exp_data in experiment_neutron_data.items():
    if ExperimentDataKey.FOM_RESULTS not in exp_data:
        print(f"No good fit data on Experiment {exp_id}")
        continue

    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]

    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()

    exp_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED].copy()
    borders = exp_data[ExperimentDataKey.BORDERS]

    psd_report = proc.classify(
        psd_report,
        calibrated_energy_column,
        borders,
        DetectorDataframeColumn.NEW_N_CLASS
    )

    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

### Neutron/Gamma Separation

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    n_class_col_name = DetectorDataframeColumn.NEW_N_CLASS.value
    
    gamma_only = psd_report.query(f"~{n_class_col_name}").copy()
    neutrons_only = psd_report.query(n_class_col_name).copy()
    exp_data[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only
    exp_data[ExperimentDataKey.GAMMA_ONLY] = gamma_only

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
    gamma_only = exp_data[ExperimentDataKey.GAMMA_ONLY]
    signals_df = exp_data["signals_df"]
    
    neutron_signals = signals_df.loc[neutrons_only.index]
    gamma_signals = signals_df.loc[gamma_only.index]

    exp_data["neutron_signals"] = neutron_signals
    exp_data["gamma_signals"] = gamma_signals

### Figure 8a Processing

In [ ]:
exp_id = "TB-26"
start = 7
step = 20
count = 5
bar_psd_margin = 0.0025

In [ ]:
# exp_data = experiment_neutron_data[exp_id]
# fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
# fom_results.head(20)

In [ ]:
end = start + count*step
x_idxs = range(start, end, step)

In [ ]:
exp_data = experiment_neutron_data[exp_id]
plot_data = figure_data["a"]
counts = exp_data[ExperimentDataKey.PSD_HISTOGRAM].T
x_edges = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
y_edges = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]

_x = x_edges[:-1]
_y = y_edges[:-1] + bar_psd_margin
_xw = x_edges[1:] - x_edges[:-1]
_yw = y_edges[1:] - y_edges[:-1] - bar_psd_margin

_xs = _x[x_idxs]
_xsw = _xw[x_idxs]
counts = counts[:, x_idxs]
# _xs = _x
# _xsw = _xw
print(counts.shape)

_xx, _yy = np.meshgrid(_xs, _y)
x, y = np.ravel(_xx), np.ravel(_yy)
_xxw, _yyw = np.meshgrid(_xsw, _yw)
dx, dy = np.ravel(_xxw), np.ravel(_yyw)
dz = np.ravel(counts)
z = np.zeros_like(dz)

height_mask = dz > 10
masked_vals = [val[height_mask] for val in [x, y, z, dx, dy, dz]]
x, y, z, dx, dy, dz = masked_vals

fig_8a_plot_data = (x, y, z, dx, dy, dz)
plot_data["bar_plot_data"] = fig_8a_plot_data

In [ ]:
exp_data = experiment_neutron_data[exp_id]
plot_data = figure_data["a"]
fom_df = exp_data[ExperimentDataKey.FOM_RESULTS]
slice_fom_df = fom_df.iloc[x_idxs]
plot_data["slice_fom_df"] = slice_fom_df

### Figure 8b Processing

In [ ]:
exp_id = "TB-26"
chosen_idx = 50

In [ ]:
exp_data = experiment_neutron_data[exp_id]
plot_data = figure_data["b"]
counts = exp_data[ExperimentDataKey.PSD_HISTOGRAM].T
y_edges = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
fom_df = exp_data[ExperimentDataKey.FOM_RESULTS]

chosen_slice_x = (y_edges[1:] + y_edges[:-1]) / 2
chosen_slice_widths = y_edges[1:] - y_edges[:-1]
chosen_slice_y = counts[:, chosen_idx]
chosen_slice_fom_row = fom_df.loc[chosen_idx]

keys = ["mu1", "sigma1", "a1", "mu2", "sigma2", "a2"]
bimodal_params = [chosen_slice_fom_row[key] for key in keys]

plot_data["chosen_slice_data"] = (chosen_slice_x, chosen_slice_y, chosen_slice_widths)
plot_data["bimodal_params"] = bimodal_params

## Plotting

### Plot Style Constants

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"

### Plot Functions

In [ ]:
def plot_figure_8a(ax: mpl.axes.Axes, elev: float, azim: float):
    fig_data = figure_data["a"]
    plot_data = fig_data["bar_plot_data"]
    slice_fom_df = fig_data["slice_fom_df"]
    dz = plot_data[-1]

    cmap = plt.colormaps["viridis"]
    min_dz = 0
    max_dz = np.max(dz)
    norm = mpl.colors.Normalize(vmin=min_dz, vmax=max_dz)
    mapped_colors = [cmap(norm(dz_val)) for dz_val in dz]

    ax.view_init(elev=elev, azim=azim)
    ax.bar3d(
        *plot_data,
        color=mapped_colors,
        shade=False,
        zsort="max",
        axlim_clip=True,
        edgecolors="black",
        linewidths=0.1
    )

    for slice_row in slice_fom_df.itertuples():
        slice_height_min = slice_row.slice_energy_min
        slice_height_max = slice_row.slice_energy_max
        slice_height_mid = (slice_height_min + slice_height_max) / 2
        slice_fom = slice_row.fom

        bimodal_x = np.linspace(0, 0.5, 100)
        bimodal_z = proc.bimodal(
            bimodal_x,
            slice_row.mu1,
            slice_row.sigma1,
            slice_row.a1,
            slice_row.mu2,
            slice_row.sigma2,
            slice_row.a2
        )
        
        ax.text(
            slice_height_max - 0, 0.02, 0, f"FOM={slice_fom:.2f}",
            zdir="y",
            ha="left",
            va="center",
            fontsize = 0.75 * fontsize,
            fontweight = "bold"
        )
        ax.plot(bimodal_x, bimodal_z, zs=slice_height_max, zdir="x", lw=3, color="black", zorder=1)

    ax.set_xlim(0, 2000)
    ax.set_ylim(0, 0.5)
    ax.set_zlim(0, None)
    ax.set_box_aspect((1, 1, 1), zoom=0.85)

    ax.set_xlabel("Pulse integral (ADC channels)", fontsize=fontsize)
    ax.set_ylabel("PSD", fontsize=fontsize)
    ax.set_zlabel("Counts (x1000)", fontsize=fontsize)
    ax.tick_params(labelsize=fontsize, pad=0)

    ax.zaxis.set_major_locator(mpl.ticker.MaxNLocator(5))
    ax.zaxis.set_major_formatter(lambda z, loc: f"{z / 1000:.1f}" if loc != 0 else "")
    ax.xaxis.labelpad = 1.5 * fontsize
    ax.yaxis.labelpad = 1 * fontsize
    ax.zaxis.labelpad = 0.7 * fontsize
    # ax.tick_params("x", pad=0)
    # ax.tick_params("y", pad=0)
    ax.tick_params("z", pad=-5)
    x_ticklabels = ax.xaxis.get_ticklabels()
    for ticklabel in x_ticklabels:
        ticklabel.set_ha("right")
        ticklabel.set_va("center")
    y_ticklabels = ax.yaxis.get_ticklabels()
    for ticklabel in y_ticklabels:
        ticklabel.set_ha("center")
        ticklabel.set_va("top")
    z_ticklabels = ax.zaxis.get_ticklabels()
    for ticklabel in z_ticklabels:
        ticklabel.set_ha("left")
        ticklabel.set_va("top")
    ax.xaxis.set_pane_color((1, 1, 1, 0))
    ax.yaxis.set_pane_color((1, 1, 1, 0))
    ax.zaxis.set_pane_color((1, 1, 1, 0))

In [ ]:
def plot_figure_8b(ax: mpl.axes.Axes):
    fig_data = figure_data["b"]
    chosen_slice_data = fig_data["chosen_slice_data"]
    bimodal_params = fig_data["bimodal_params"]
    slice_x, slice_y, slice_w = chosen_slice_data
    mu1, sigma1, _, mu2, sigma2, *_ = bimodal_params

    bimodal_x = np.linspace(0, 0.6, 1000)
    bimodal_y = proc.bimodal(bimodal_x, *bimodal_params)
    gamma_vline_xs = [mu1, mu1 + sigma1]
    neutron_vline_xs = [mu2, mu2 + sigma2]
    # # gamma_vline_labels = [r"${\mu}_{\gamma}$", r"${\mu}_{\gamma} + 5{\sigma}_{\gamma}$"]
    # # neutron_vline_labels = [r"${\mu}_{n}$", r"${\mu}_{n} + 5{\sigma}_{n}$"]
    # # gamma_ymax = [proc.gaussian(x, mu1, sigma1, a1) for x in gamma_vline_xs]
    # # neutron_ymax = [proc.gaussian(x, mu2, sigma2, a2) for x in neutron_vline_xs]
    # # print(gamma_ymax)
    # # print(gamma_ymax + neutron_ymax)

    ax.bar(slice_x, slice_y, width=slice_w, align="center", color=bg_blue)
    ax.plot(bimodal_x, bimodal_y, lw=3, color=bg_grey)
    # ax.plot(gauss_y, gamma_gaussian, lw=5, color=bg_grey)
    # ax.plot(gauss_y, neutron_gaussian, lw=5, color=bg_grey)
    # ax.plot(ymids, Z_slice, "-", lw=2, color="black", alpha=0.5)

    # # trans = ax.transData + ax.transAxes.inverted()
    # # trans = mpl.transforms.blended_transform_factory(ax.transData, ax.transAxes)
    ax.axvline(0, 0, 0, alpha=0)  # "burner" line - first line never transforms properly (transform not initiated?)
    for vline_x in gamma_vline_xs + neutron_vline_xs:
        ax.axvline(vline_x, 0, 1, lw=2, color="black")
    #     # ax.text(vline_x + 0.005, 3100, vline_label, fontsize=fontsize-10)
    margin = 0.005
    text_y = 790
    ax.text(
        mu1 - margin,
        text_y,
        r"${\mu}_{\gamma}$",
        fontsize=fontsize-5,
        ha="right",
        va="top"
    )
    ax.text(
        (mu1 + sigma1) + margin,
        text_y,
        r"${\sigma}_{\gamma}$",
        fontsize=fontsize-5,
        ha="left",
        va="top"
    )
    ax.text(
        mu2 - margin,
        text_y,
        r"${\mu}_{n}$",
        fontsize=fontsize-5,
        ha="right",
        va="top"
    )
    ax.text(
        (mu2 + sigma2) + margin,
        text_y,
        r"${\sigma}_{n}$",
        fontsize=fontsize-5,
        ha="left",
        va="top"
    )

    ax.set_xlabel("PSD", fontsize=fontsize)
    ax.set_ylabel("Counts", fontsize=fontsize)
    ax.set_xlim(0, 0.5)
    # ax.set_ylim(0, 800)
    # ax.yaxis.set_major_formatter(lambda x, _: f"{x / 1000:.1f}")
    ax.yaxis.set_major_formatter(lambda x, _: "" if x == 0 else f"{x:.1f}")
    ax.tick_params(labelsize=fontsize)
    # ax.tick_params("x", pad=8)

### Plot Creation

In [ ]:
fig, ax = plt.subplots(
    figsize=(10, 10),
    subplot_kw={"projection": "3d"},
    layout="constrained"
)
plot_figure_8a(ax, 30, -30)

In [ ]:
fig, ax = plt.subplots(
    layout="constrained"
)
plot_figure_8b(ax)

## Done

In [ ]:
input("Processing done, hit Enter to finish")
helpers.stop()